# Crypto.Net · 01 — Wallets and keys

*Built by Gravicode Studios, led by Kang Fadhil.*

> **Safe to run.** This notebook only reads public networks and signs offline; it never broadcasts a transaction. Keys created here are demo keys — never send real funds to them.

Open in VS Code with the **Polyglot Notebooks** extension (or Jupyter with .NET Interactive) and run the cells in order. [Versi Bahasa Indonesia](./01-wallet-and-keys.id.ipynb)

What you will do: generate a BIP-39 recovery phrase, derive accounts on five chains from it, export extended public keys, encrypt a key into a Web3 keystore, and compare the Rust core with the managed C# engine.

In [ ]:
#r "nuget: Crypto.Net.Extensions, 1.0.0"

In [ ]:
using Crypto.Net.Core;
using Crypto.Net.Native;
using Crypto.Net.Wallet;
using Crypto.Net.Evm;
using Crypto.Net.Bitcoin;
using Crypto.Net.Solana;
using Crypto.Net.Polkadot;
using Crypto.Net.Cosmos;
using Crypto.Net.Extensions;

$"Rust core loaded: {NativeLoader.IsNativeAvailable} · ABI 0x{NativeLoader.AbiVersion:X8} · {NativeLoader.RuntimeIdentifier}"

## Recovery phrase (BIP-39)

12–24 words encode 128–256 bits of entropy plus a checksum. `Mnemonic.Validate` checks both the wordlist and the checksum.

In [ ]:
var phrase = Mnemonic.Generate(12);
Console.WriteLine(phrase);
Console.WriteLine($"valid: {Mnemonic.Validate(phrase)}");
Console.WriteLine($"one wrong word: {Mnemonic.Validate(phrase.Replace(phrase.Split(' ')[0], "zoo"))}");

The rest of this notebook uses the public BIP-39 test phrase so the results are reproducible — and so you can compare them with MetaMask, Phantom, Polkadot.js or Keplr.

In [ ]:
const string TestPhrase = "abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon about";
byte[] seed = Mnemonic.ToSeed(TestPhrase);
Convert.ToHexStringLower(seed)

## One phrase, every chain

`HdWallet` holds three roots: BIP-32 (secp256k1) for EVM, Bitcoin and Cosmos, SLIP-10 (ed25519) for Solana, and substrate-bip39 for Polkadot.

In [ ]:
var wallet = HdWallet.FromMnemonic(TestPhrase);

var eth  = wallet.GetEvmAccount(0, EvmChain.Ethereum);
var btc  = wallet.GetBitcoinAccount(BitcoinAddressType.SegWitP2WPKH);
var tap  = wallet.GetBitcoinAccount(BitcoinAddressType.TaprootP2TR);
var sol  = wallet.GetSolanaAccount(0);
var dot  = wallet.GetPolkadotAccount("", PolkadotChain.Polkadot);
var atom = wallet.GetCosmosAccount(0);

new[]
{
    new { Chain = "Ethereum",          Path = eth.DerivationPath,  Address = eth.Address.Value },
    new { Chain = "Bitcoin (SegWit)",  Path = btc.DerivationPath,  Address = btc.Address.Value },
    new { Chain = "Bitcoin (Taproot)", Path = tap.DerivationPath,  Address = tap.Address.Value },
    new { Chain = "Solana",            Path = sol.DerivationPath,  Address = sol.Address.Value },
    new { Chain = "Polkadot",          Path = "(root, sr25519)",   Address = dot.Address.Value },
    new { Chain = "Cosmos Hub",        Path = atom.DerivationPath, Address = atom.Address.Value },
}

## Extended keys and custom paths

Account-level `zpub`/`xpub` let a watch-only wallet derive receive addresses without the private keys.

In [ ]:
Console.WriteLine($"BIP-84 zpub: {wallet.GetBitcoinAccountXpub(BitcoinAddressType.SegWitP2WPKH)}");

var key = wallet.DerivePath("m/44'/60'/0'/0/7");
Console.WriteLine($"m/44'/60'/0'/0/7 → {EvmAddress.FromPrivateKey(key.PrivateKey.Span)}");
Console.WriteLine($"depth {key.Depth}, parent fingerprint {key.ParentFingerprint:x8}");
key.Dispose();

## Encrypted keystore (Web3 Secret Storage v3)

The same format geth and MetaMask import. `KeyStoreKdf.Light` keeps this demo fast; use `KeyStoreKdf.Standard` in real applications.

In [ ]:
string json = eth.ExportKeyStore("correct horse battery staple", KeyStoreKdf.Light);
Console.WriteLine(json);

var restored = EvmAccount.FromKeyStore(json, "correct horse battery staple");
Console.WriteLine($"restored {restored.Address} — same account: {restored.Address == eth.Address}");
try { KeyStore.Decrypt(json, "wrong password"); }
catch (System.Security.Cryptography.CryptographicException e) { Console.WriteLine($"wrong password → {e.Message}"); }

## Two engines, identical bytes

Every primitive runs in Rust or in managed C#. `UseBackend` switches for the current flow only; the outputs are byte-identical.

In [ ]:
byte[] data = "Crypto.Net"u8.ToArray();
string rust, managed;
using (CryptoNative.UseBackend(CryptoBackend.Native))  rust    = Convert.ToHexStringLower(CryptoNative.Keccak256(data));
using (CryptoNative.UseBackend(CryptoBackend.Managed)) managed = Convert.ToHexStringLower(CryptoNative.Keccak256(data));
Console.WriteLine($"{rust}\n{managed}\nidentical: {rust == managed}");

CryptoBenchmark.Run(scale: 0.1)
    .Select(r => new { r.Name, Rust = Math.Round(r.NativeOpsPerSec), Managed = Math.Round(r.ManagedOpsPerSec), SpeedUp = Math.Round(r.Speedup, 1) })

Dispose secrets when you are done.

In [ ]:
foreach (IDisposable d in new IDisposable[] { eth, btc, tap, sol, dot, atom, restored, wallet }) d.Dispose();
"disposed"